# Notebook 02 — Traitement distribué avec PySpark
## Atelier 2 (1/2) — Préparer PySpark
Session Spark en mode local `local[*]` (tous les cœurs du poste), connecteur MongoDB pour Spark 4 :
`org.mongodb.spark:mongo-spark-connector_2.13:11.1.0` (Scala 2.13, requis par Spark 4.0).

In [ ]:
import os, sys
os.environ["PYSPARK_PYTHON"] = sys.executable                 # Windows : Spark utilise le Python du kernel
os.environ.setdefault("SPARK_LOCAL_HOSTNAME", "localhost")     # Windows : nom du PC avec accent
os.environ["HADOOP_HOME"] = r"C:\hadoop"                       # Windows : winutils.exe
os.environ["PATH"] = r"C:\hadoop\bin;" + os.environ["PATH"]    # Windows : hadoop.dll   # Windows : hadoop.dll
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("BigDataCommerceAnalytics")
    .master("local[*]")
    .config("spark.jars.packages", "org.mongodb.spark:mongo-spark-connector_2.13:11.1.0")
    .config("spark.mongodb.read.connection.uri", "mongodb://localhost:27017")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print("PySpark      :", pyspark.__version__)
print("Spark        :", spark.version)
print("Mode         :", spark.sparkContext.master)
print("Cœurs utilisés (parallélisme par défaut) :", spark.sparkContext.defaultParallelism)
print("Spark UI     :", spark.sparkContext.uiWebUrl)

PySpark      : 4.0.1
Spark        : 4.0.1
Mode         : local[*]
Cœurs utilisés (parallélisme par défaut) : 16
Spark UI     : http://localhost:4040


## Atelier 2 (2/2) — 1. Lire les données depuis MongoDB

In [2]:
def lire_collection(nom):
    return (spark.read.format("mongodb")
            .option("database", "commerce")
            .option("collection", nom)
            .load())

df_commandes = lire_collection("raw_commandes")
df_payments  = lire_collection("raw_payments")

print("=== raw_commandes ===")
df_commandes.printSchema()
df_commandes.show(5, truncate=False)

print("=== raw_payments ===")
df_payments.printSchema()
df_payments.show(5, truncate=False)

print("Documents lus — commandes :", df_commandes.count(), "| paiements :", df_payments.count())

=== raw_commandes ===
root
 |-- _id: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- cart_discounted_total: double (nullable = true)
 |-- cart_id: integer (nullable = true)
 |-- cart_total: double (nullable = true)
 |-- category: string (nullable = true)
 |-- discount_percentage: double (nullable = true)
 |-- discounted_line_total: double (nullable = true)
 |-- line_total: double (nullable = true)
 |-- order_date: timestamp (nullable = true)
 |-- order_id: string (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- product_title: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- user_id: integer (nullable = true)
 |-- username: string (nullable = true)

+------------------------+--------------+---------------------+-------+----------+------------------+-------------------+---------------------+-----------------+-------------------+----------+----------+----------------------+--------+--------